In [ ]:
!apt-get update
!apt install -y chromium-chromedriver
!pip install selenium


In [ ]:
!apt-get purge chromium-browser
!apt-get update
!apt install -y wget unzip
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install -y

# Install ChromeDriver
!wget -N https://chromedriver.storage.googleapis.com/114.0.5735.90/chromedriver_linux64.zip
!unzip chromedriver_linux64.zip
!mv chromedriver /usr/local/bin/

In [ ]:
!unzip /content/chromedriver_linux64.zip


In [ ]:
# Import necessary libraries
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from google.colab import auth
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import drive


In [ ]:
# Authenticate and mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Set the path to your image folder inside Google Drive
IMAGE_FOLDER = "/content/drive/MyDrive/LGBTQ Memes/1"

In [ ]:
# Set up Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.binary_location = "/usr/bin/google-chrome"  # Specify the Chrome binary location

# Set up ChromeDriver path
chrome_driver_path = "/content/chromedriver"
service = Service(chrome_driver_path)

In [ ]:
# Uninstall old ChromeDriver (if any)
!apt-get remove -y chromium-chromedriver

# Install the latest Google Chrome
!wget -O chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i chrome.deb
!apt-get install -f  # Fix dependencies

# Verify Chrome installation
!google-chrome --version

# Fetch ChromeDriver version that matches Chrome
chrome_version = !google-chrome --version
chrome_version = chrome_version[0].split()[2]  # Extract version number

# Get only major version (e.g., 133 from 133.0.6943.98)
major_version = chrome_version.split(".")[0]

# Download the matching ChromeDriver
!wget -O chromedriver.zip https://storage.googleapis.com/chrome-for-testing-public/{major_version}/linux64/chromedriver-linux64.zip
!unzip chromedriver.zip
!chmod +x chromedriver-linux64/chromedriver
!mv chromedriver-linux64/chromedriver /usr/bin/chromedriver

# Verify ChromeDriver installation
!chromedriver --version


In [ ]:
!google-chrome --version


In [ ]:
!pip install selenium webdriver-manager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Set up Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Automatically download & set up the correct ChromeDriver
service = Service(ChromeDriverManager().install())

# Start WebDriver
driver = webdriver.Chrome(service=service, options=chrome_options)

print("✅ WebDriver successfully initialized!")


In [ ]:
# Start WebDriver with updated setup
driver = webdriver.Chrome(service=service, options=chrome_options)

In [ ]:
# Function to perform reverse image search
def reverse_image_search(image_path):
    driver.get("https://www.google.com/imghp")

    # Click on the Google Lens button
    time.sleep(2)
    lens_button = driver.find_element(By.CSS_SELECTOR, "div[jsname='R5eUCe']")
    lens_button.click()

    # Upload image
    time.sleep(2)
    upload_input = driver.find_element(By.CSS_SELECTOR, "input[type='file']")
    upload_input.send_keys(image_path)

    # Wait for results
    time.sleep(5)

    # Extract top search result links
    results = driver.find_elements(By.CSS_SELECTOR, "div.BVG0Nb a")
    urls = [result.get_attribute("href") for result in results[:3]]  # Get top 3 links

    return urls if urls else ["No Match Found"]

In [ ]:
# Process all images in folder
image_files = [os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER) if f.endswith((".jpg", ".png", ".jpeg"))]
output_data = []

for img_path in image_files:
    print(f"Processing: {img_path}")
    urls = reverse_image_search(img_path)
    output_data.append({"Image": os.path.basename(img_path), "Top Matches": ", ".join(urls)})

# Save results as CSV
df = pd.DataFrame(output_data)
df.to_csv("/content/reverse_image_search_results.csv", index=False)

# Move CSV to Google Drive
!mv /content/reverse_image_search_results.csv /content/drive/MyDrive/
print("✅ Reverse image search complete! Results saved in Google Drive.")